# LAB3 - Bài 4 (Trang 9): Neural Network trên Iris với k-fold cross validation (k=10)
**Đinh Nhật Huyền Nhân - 6451071055 - CQ.64.CNTT**

In [1]:
print("Đinh Nhật Huyền Nhân - 6451071055")
# a) Đọc bộ dữ liệu Iris từ sklearn
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
import numpy as np
import pandas as pd
import tensorflow as tf
tf.get_logger().setLevel("ERROR")
from sklearn.datasets import load_iris
iris = load_iris()
X, Y = iris.data, iris.target
print("X:", X.shape, "| Y:", Y.shape, "| Các lớp:", iris.target_names.tolist())

Đinh Nhật Huyền Nhân - 6451071055


X: (150, 4) | Y: (150,) | Các lớp: ['setosa', 'versicolor', 'virginica']


In [2]:
print("Đinh Nhật Huyền Nhân - 6451071055")
# b) Chuẩn hóa dữ liệu về [0,1]
X_max = X.max(axis=0)
X = X / X_max
print("Min:", X.min(axis=0)); print("Max:", X.max(axis=0))

Đinh Nhật Huyền Nhân - 6451071055
Min: [0.5443038  0.45454545 0.14492754 0.04      ]
Max: [1. 1. 1. 1.]


In [3]:
print("Đinh Nhật Huyền Nhân - 6451071055")
# c) Hàm create_model: Neural Network tương tự câu 1.h + biên dịch
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

def create_model():
    model = Sequential([
        Input(shape=(4,)),
        Dense(10, activation="relu"),
        Dense(20, activation="relu"),
        Dense(3, activation="softmax"),
    ])
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

create_model().summary()

Đinh Nhật Huyền Nhân - 6451071055


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 10)             │            50 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 20)             │           220 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 3)              │            63 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 333 (1.30 KB)

 Trainable params: 333 (1.30 KB)

 Non-trainable params: 0 (0.00 B)

In [4]:
print("Đinh Nhật Huyền Nhân - 6451071055")
# d) Huấn luyện bằng k-fold cross validation (k=10); in accuracy, precision, recall, f1 từng fold
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
tf.keras.utils.set_random_seed(42)

kfold = KFold(n_splits=10, shuffle=True, random_state=42)
models = []
acc_list, pre_list, rec_list, f1_list = [], [], [], []
pre_cls, rec_cls, f1_cls = [], [], []        # theo từng class

for fold, (train_index, val_index) in enumerate(kfold.split(X), start=1):
    model = create_model()
    model.fit(X[train_index], Y[train_index], epochs=100, batch_size=1, verbose=0)
    models.append(model)                     # lưu lại mô hình sau mỗi fold
    y_pred = np.argmax(model.predict(X[val_index], verbose=0), axis=1)
    y_true = Y[val_index]
    acc = accuracy_score(y_true, y_pred)
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    pc, rc, fc, _ = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1, 2], average=None, zero_division=0)
    acc_list.append(acc); pre_list.append(p); rec_list.append(r); f1_list.append(f)
    pre_cls.append(pc); rec_cls.append(rc); f1_cls.append(fc)
    print(f"Fold {fold:2d}: accuracy = {acc:.4f} | precision = {p:.4f} | recall = {r:.4f} | f1-score = {f:.4f}")

Đinh Nhật Huyền Nhân - 6451071055


Fold  1: accuracy = 1.0000 | precision = 1.0000 | recall = 1.0000 | f1-score = 1.0000


Fold  2: accuracy = 1.0000 | precision = 1.0000 | recall = 1.0000 | f1-score = 1.0000


Fold  3: accuracy = 1.0000 | precision = 1.0000 | recall = 1.0000 | f1-score = 1.0000


Fold  4: accuracy = 0.9333 | precision = 0.9524 | recall = 0.9333 | f1-score = 0.9373


Fold  5: accuracy = 1.0000 | precision = 1.0000 | recall = 1.0000 | f1-score = 1.0000


Fold  6: accuracy = 0.9333 | precision = 0.9167 | recall = 0.9444 | f1-score = 0.9221


Fold  7: accuracy = 0.8667 | precision = 0.9167 | recall = 0.8333 | f1-score = 0.8413


Fold  8: accuracy = 1.0000 | precision = 1.0000 | recall = 1.0000 | f1-score = 1.0000


Fold  9: accuracy = 1.0000 | precision = 1.0000 | recall = 1.0000 | f1-score = 1.0000


Fold 10: accuracy = 1.0000 | precision = 1.0000 | recall = 1.0000 | f1-score = 1.0000


In [5]:
print("Đinh Nhật Huyền Nhân - 6451071055")
# e) Trung bình các độ đo sau k-fold
print(f"Average Accuracy : {np.mean(acc_list):.4f}")
print(f"Average Recall   : {np.mean(rec_list):.4f}")
print(f"Average Precision: {np.mean(pre_list):.4f}")
print(f"Average F1-score : {np.mean(f1_list):.4f}")

Đinh Nhật Huyền Nhân - 6451071055
Average Accuracy : 0.9733
Average Recall   : 0.9711
Average Precision: 0.9786
Average F1-score : 0.9701


In [6]:
print("Đinh Nhật Huyền Nhân - 6451071055")
# f) Trung bình Recall, Precision, F1-score cho từng class
bang_cls = pd.DataFrame({
    "Class": iris.target_names,
    "Average Recall": np.mean(rec_cls, axis=0),
    "Average Precision": np.mean(pre_cls, axis=0),
    "Average F1-score": np.mean(f1_cls, axis=0),
})
print(bang_cls.round(4).to_string(index=False))

Đinh Nhật Huyền Nhân - 6451071055
     Class  Average Recall  Average Precision  Average F1-score
    setosa          1.0000             1.0000            1.0000
versicolor          0.9333             0.9857            0.9499
 virginica          0.9800             0.9500            0.9603


In [7]:
print("Đinh Nhật Huyền Nhân - 6451071055")
# g) Dùng mô hình có F1-score tốt nhất để dự đoán 03 mẫu mới
best = int(np.argmax(f1_list))
best_model = models[best]
print(f"Mô hình tốt nhất: fold {best + 1} (F1-score = {f1_list[best]:.4f})")
mau_moi = np.array([[6.2, 2.9, 4.3, 1.3],
                    [5.1, 3.5, 1.4, 0.2],
                    [7.3, 2.8, 6.4, 2.1]])
prob = best_model.predict(mau_moi / X_max, verbose=0)
for m, pr in zip(mau_moi, prob):
    print(f"{m} -> {iris.target_names[np.argmax(pr)]}  (xác suất: {np.round(pr, 3)})")

Đinh Nhật Huyền Nhân - 6451071055
Mô hình tốt nhất: fold 1 (F1-score = 1.0000)
[6.2 2.9 4.3 1.3] -> versicolor  (xác suất: [0. 1. 0.])
[5.1 3.5 1.4 0.2] -> setosa  (xác suất: [1. 0. 0.])
[7.3 2.8 6.4 2.1] -> virginica  (xác suất: [0. 0. 1.])
